# The DSLD loader

Each cell is one call.

In [1]:
import re
from engine.repositories.dsld import loader

raw_labels = loader.load_raw(n=1000, order="random", seed=42)

Cell 1 — Collect ingredient amounts
Cell 2 — Convert mass units to milligrams
Cell 3 — Keep positive, exact mass amounts with a serving reference
Cell 4 — Find the largest amount within each label and serving
Cell 5 — Produce one ingredient name per label
Cell 6 — Mark names that represent blend totals
Cell 7 — Display the names

In [18]:
import pandas as pd

amounts = pd.DataFrame(
    [
        {
            "dsld_id": raw["id"],
            "ingredient": ingredient.get("name"),
            "category": ingredient.get("category"),
            "serving": quantity.get("servingSizeOrder"),
            "amount": quantity.get("quantity"),
            "unit": quantity.get("unit"),
            "operator": quantity.get("operator"),
        }
        for raw in raw_labels
        for ingredient in raw.get("ingredientRows") or []
        for quantity in ingredient.get("quantity") or []
    ],
    columns=[
        "dsld_id", "ingredient", "category",
        "serving", "amount", "unit", "operator",
    ],
)

amounts.head()

,dsld_id,ingredient,category,serving,amount,unit,operator
0,228731,Calories,other,1,5.0,Calorie(s),=
1,228731,Total Carbohydrates,sugar,1,1.0,Gram(s),<
2,228731,Niacin,vitamin,1,60.0,mg,=
3,228731,Vitamin B6,vitamin,1,1.0,mg,=
4,228731,Folic Acid,vitamin,1,500.0,mcg,=


In [19]:
to_mg = {
    "g": 1000,
    "gram(s)": 1000,
    "grams": 1000,
    "mg": 1,
    "mcg": 0.001,
    "µg": 0.001,
    "μg": 0.001,
    "ng": 0.000001,
}

amounts["mg"] = (
    pd.to_numeric(amounts["amount"], errors="coerce")
    * amounts["unit"].fillna("").str.strip().str.lower().map(to_mg)
)

In [20]:
comparable = amounts.loc[
    amounts["operator"].eq("=")
    & amounts["mg"].gt(0)
    & amounts["serving"].notna()
].copy()

In [21]:
maximum = comparable.groupby(
    ["dsld_id", "serving"]
)["mg"].transform("max")

winners = comparable.loc[comparable["mg"].eq(maximum)]

winners

,dsld_id,ingredient,category,serving,amount,unit,operator,mg
6,228731,Citrulline Malate,non-nutrient/non-botanical,1,6.00,Gram(s),=,6000.0
12,251040,Chromium,mineral,1,200.00,mcg,=,0.2
16,62293,FoodState Farm Fresh Fruit Blend,blend,1,10.00,mg,=,10.0
18,247050,Total Fat,fat,1,1.00,Gram(s),=,1000.0
21,303452,Total Fat,fat,1,2.50,Gram(s),=,2500.0
...,...,...,...,...,...,...,...,...
5321,295754,Proprietary Blend,blend,1,540.00,mg,=,540.0
5322,311749,Pygeum africanum bark extract,botanical,1,500.00,mg,=,500.0
5327,204982,Greens Blend,blend,1,11.25,Gram(s),=,11250.0
5328,306081,Damiana extract,botanical,1,400.00,mg,=,400.0


In [22]:
candidates = winners.groupby("dsld_id")["ingredient"].agg(
    lambda names: list(dict.fromkeys(names.dropna()))
)

result = pd.DataFrame([
    {
        "dsld_id": raw["id"],
        "product": raw.get("fullName"),
    }
    for raw in raw_labels
])

result["candidates"] = (
    result["dsld_id"]
    .map(candidates)
    .apply(lambda value: value if isinstance(value, list) else [])
)

result["dominant_declared_name"] = result["candidates"].apply(
    lambda names: names[0] if len(names) == 1 else None
)

In [23]:
blend_ids = set(
    winners.loc[winners["category"].eq("blend"), "dsld_id"]
)

result["is_blend"] = (
    result["dominant_declared_name"].notna()
    & result["dsld_id"].isin(blend_ids)
)

In [24]:
result[[
    "dsld_id",
    "product",
    "dominant_declared_name",
    "is_blend",
]]

,dsld_id,product,dominant_declared_name,is_blend
0,228731,C4 Ultimate On The Go Watermelon,Citrulline Malate,False
1,251040,Chromium Picolinate 200 mcg,Chromium,False
2,279344,Carline Thistle,NaN,False
3,62293,One Daily,FoodState Farm Fresh Fruit Blend,True
4,247050,DHA 710 Select,Total Fat,False
...,...,...,...,...
995,295754,Insinase,Proprietary Blend,True
996,311749,Pygeum Africanum Extract,Pygeum africanum bark extract,False
997,204982,Juiced Up Bangin Berries,Greens Blend,True
998,306081,FemGasm,Damiana extract,False


In [25]:
servings = pd.DataFrame([
    {
        "dsld_id": raw["id"],
        "serving": size.get("order"),
        "minimum": size.get("minQuantity"),
        "maximum": size.get("maxQuantity"),
        "unit": size.get("unit"),
    }
    for raw in raw_labels
    for size in raw.get("servingSizes") or []
])

In [26]:
servings["total_serving_mg"] = (
    pd.to_numeric(servings["minimum"], errors="coerce")
    * servings["unit"].fillna("").str.strip().str.lower().map(to_mg)
)

serving_weights = servings.loc[
    servings["minimum"].eq(servings["maximum"])
    & servings["total_serving_mg"].gt(0),
    ["dsld_id", "serving", "total_serving_mg"],
]

In [27]:
dominant_amounts = winners.merge(
    result[["dsld_id", "dominant_declared_name"]].dropna(),
    left_on=["dsld_id", "ingredient"],
    right_on=["dsld_id", "dominant_declared_name"],
    how="inner",
)

dominant_amounts = dominant_amounts.merge(
    serving_weights,
    on=["dsld_id", "serving"],
    how="left",
    validate="many_to_one",
)

In [28]:
dominant_amounts["percent_of_serving"] = (
    100
    * dominant_amounts["mg"]
    / dominant_amounts["total_serving_mg"]
)

dominant_amounts["needs_review"] = (
    dominant_amounts["percent_of_serving"] > 100
)

In [29]:
dominant_amounts[[
    "dsld_id",
    "ingredient",
    "category",
    "serving",
    "mg",
    "total_serving_mg",
    "percent_of_serving",
    "needs_review",
]]

,dsld_id,ingredient,category,serving,mg,total_serving_mg,percent_of_serving,needs_review
0,228731,Citrulline Malate,non-nutrient/non-botanical,1,6000.0,NaN,NaN,False
1,251040,Chromium,mineral,1,0.2,NaN,NaN,False
2,62293,FoodState Farm Fresh Fruit Blend,blend,1,10.0,NaN,NaN,False
3,247050,Total Fat,fat,1,1000.0,NaN,NaN,False
4,303452,Total Fat,fat,1,2500.0,NaN,NaN,False
...,...,...,...,...,...,...,...,...
860,295754,Proprietary Blend,blend,1,540.0,NaN,NaN,False
861,311749,Pygeum africanum bark extract,botanical,1,500.0,500.0,100.0,False
862,204982,Greens Blend,blend,1,11250.0,12500.0,90.0,False
863,306081,Damiana extract,botanical,1,400.0,NaN,NaN,False


## load_raw — records from the dump

In [13]:
ingredient_texts = []
stack = list(reversed(raw_labels[3].get("ingredientRows") or []))

while stack:
    ingredient = stack.pop()

    for field in ("notes", "description", "text"):
        text = ingredient.get(field)
        if isinstance(text, str) and text.strip():
            ingredient_texts.append(text.strip())

    stack.extend(reversed(ingredient.get("nestedRows") or []))

print("\n\n".join(ingredient_texts) or "No ingredient text found.")

Vitamin A (Form: as Alpha Carotene, & Beta Carotene, with Mixed Carotenoids (Form: Cryptoxanthin, Lutein, Zeaxanthin))

organic Orange Note: Includes certified organic whole oranges

Vitamin E (Form: full spectrum of Mixed Tocotrienols, and Tocopherols)

organic whole Orange (Form: natural source Bioflavonoids)

organic Blueberry (Form: natural source Anthocyanins)

organic Cranberry (Form: natural source Proanthocyandins)


In [14]:
import pandas as pd

amounts = pd.DataFrame(
    [
        {
            "dsld_id": raw["id"],
            "ingredient": ingredient.get("name"),
            "category": ingredient.get("category"),
            "serving": quantity.get("servingSizeOrder"),
            "amount": quantity.get("quantity"),
            "unit": quantity.get("unit"),
            "operator": quantity.get("operator"),
        }
        for raw in raw_labels
        for ingredient in raw.get("ingredientRows") or []
        for quantity in ingredient.get("quantity") or []
    ],
    columns=[
        "dsld_id", "ingredient", "category",
        "serving", "amount", "unit", "operator",
    ],
)

In [15]:
to_mg = {
    "g": 1000,
    "gram(s)": 1000,
    "grams": 1000,
    "mg": 1,
    "mcg": 0.001,
    "µg": 0.001,
    "μg": 0.001,
    "ng": 0.000001,
}

amounts["mg"] = (
    pd.to_numeric(amounts["amount"], errors="coerce")
    * amounts["unit"].fillna("").str.strip().str.lower().map(to_mg)
)

In [16]:
comparable = amounts.loc[
    amounts["operator"].eq("=")
    & amounts["mg"].gt(0)
    & amounts["serving"].notna()
].copy()

In [17]:
maximum = comparable.groupby(
    ["dsld_id", "serving"]
)["mg"].transform("max")

winners = comparable.loc[comparable["mg"].eq(maximum)]

winners

,dsld_id,ingredient,category,serving,amount,unit,operator,mg
6,228731,Citrulline Malate,non-nutrient/non-botanical,1,6.00,Gram(s),=,6000.0
12,251040,Chromium,mineral,1,200.00,mcg,=,0.2
16,62293,FoodState Farm Fresh Fruit Blend,blend,1,10.00,mg,=,10.0
18,247050,Total Fat,fat,1,1.00,Gram(s),=,1000.0
21,303452,Total Fat,fat,1,2.50,Gram(s),=,2500.0
...,...,...,...,...,...,...,...,...
5321,295754,Proprietary Blend,blend,1,540.00,mg,=,540.0
5322,311749,Pygeum africanum bark extract,botanical,1,500.00,mg,=,500.0
5327,204982,Greens Blend,blend,1,11.25,Gram(s),=,11250.0
5328,306081,Damiana extract,botanical,1,400.00,mg,=,400.0


In [ ]:
raw_labels[0].get("ingredientRows")

[{'order': 1,
  'ingredientId': 284519,
  'description': None,
  'notes': None,
  'quantity': [{'servingSizeOrder': 1,
    'servingSizeQuantity': None,
    'operator': '=',
    'quantity': 5,
    'unit': 'Calorie(s)',
    'dailyValueTargetGroup': [],
    'servingSizeUnit': 'Fluid Ounce(s)'}],
  'nestedRows': [],
  'name': 'Calories',
  'category': 'other',
  'ingredientGroup': 'Calories',
  'uniiCode': '1',
  'alternateNames': [],
  'forms': []},
 {'order': 2,
  'ingredientId': 283702,
  'description': None,
  'notes': None,
  'quantity': [{'servingSizeOrder': 1,
    'servingSizeQuantity': None,
    'operator': '<',
    'quantity': 1,
    'unit': 'Gram(s)',
    'dailyValueTargetGroup': [{'name': 'Adults and children 4 or more years of age',
      'operator': '<',
      'percent': 1,
      'footnote': None}],
    'servingSizeUnit': 'Fluid Ounce(s)'}],
  'nestedRows': [{'order': 3,
    'ingredientId': 283681,
    'description': None,
    'notes': None,
    'quantity': [{'servingSizeOrder

In [4]:
texts_by_id = {
    raw["id"]: [
        statement.get("notes") or ""
        for statement in raw.get("statements") or []
    ]
    for raw in raw_labels
}

In [8]:
texts_by_id[raw_labels[2]["id"]]

['Copyright 2018 TerraVita - Distributed by ZooScape LLC, All Rights Reserved.',
 'Glycerite liquid extract (1:5)',
 'Alcohol free',
 'Suggested Use: \r\nTake 25-40 drops of extract in a small amount of warm water 3-4 times daily.\r\nShake well before using.',
 'Keep away from children.',
 'Do not use if safety seal around cap is broken or missing.',
 'No flavor']

In [3]:
loader.load_raw(709)["fullName"]

'Huperzine A'

In [4]:
loader.load_raw([889, 709])

[{'src': '01-raw/2023-02-23/data/label_160.json',
  'id': 709,
  'nhanesId': '',
  'bundleName': '',
  'fullName': 'Huperzine A',
  'brandName': 'GNC Herbal Plus Standardized',
  'brandIpSymbol': '®',
  'upcSku': '0 48107 06464 8',
  'productVersionCode': 'CODE 196311 HLG',
  'pdf': '',
  'thumbnail': '',
  'servingsPerContainer': None,
  'hasOuterCarton': False,
  'percentDvFootnote': 'Not Present',
  'labelRelationships': [{'type': 'Image difference, same product',
    'labelId': 63846},
   {'type': 'Image difference, same product', 'labelId': 74667},
   {'type': 'Image difference, same product', 'labelId': 243149}],
  'contacts': [{'contactId': 1193,
    'text': 'For more information',
    'types': ['Other'],
    'contactDetails': {'src': '01-raw/2025-09-25/data/contact_1.json',
     'id': 1193,
     'name': '',
     'streetAddress': '',
     'city': '',
     'state': '',
     'country': '',
     'zipCode': '',
     'phoneNumber': '1-888-462-2548',
     'email': '',
     'webAddress

In [5]:
[r["id"] for r in loader.load_raw([889, 709])]

[709, 889]

In [6]:
[r["id"] for r in loader.load_raw([889, 709], order="given")]

[889, 709]

In [7]:
[r["id"] for r in loader.load_raw(n=5)]

[542, 543, 544, 545, 546]

In [8]:
[r["id"] for r in loader.load_raw(n=5, order="random", seed=42)]

[228731, 251040, 279344, 62293, 247050]

## load_flattened — the pilot passages

In [9]:
labels = loader.load_flattened()
len(labels)

1000

In [10]:
labels[0]

Label(dsld_id=709, name='Huperzine A', passage='PRODUCT: Huperzine A\nBRAND: GNC Herbal Plus Standardized\nTYPE: Botanical (Capsule)\nCLAIM TYPES ON LABEL: All Other, Structure/Function\nTARGET GROUPS: Vegetarian, Adult (18 - 50 Years)\nINGREDIENTS: Huperzine A\n\nLABEL STATEMENTS:\n[Precautions re: All Other] KEEP OUT OF REACH OF CHILDREN.\n[Precautions re: Pregnant or Nursing or Prescription Medications] WARNING: Do not exceed recommended intake. Consult your physicial prior to using this product if you are pregnant, nursing, taking medication, have high blood pressure, heart problems or have a medical condition. Discontinue use two weeks prior to surgery.\n[Storage] Store in a cool, dry place.\n[General Statements: All Other Content] Supports Memory Retention*\n[Suggested/Recommended/Usage/Directions] Directions: As a dietary supplement, take one to two capsules daily.')

In [11]:
print(labels[0].passage)

PRODUCT: Huperzine A
BRAND: GNC Herbal Plus Standardized
TYPE: Botanical (Capsule)
CLAIM TYPES ON LABEL: All Other, Structure/Function
TARGET GROUPS: Vegetarian, Adult (18 - 50 Years)
INGREDIENTS: Huperzine A

LABEL STATEMENTS:
[Precautions re: All Other] KEEP OUT OF REACH OF CHILDREN.
[Precautions re: Pregnant or Nursing or Prescription Medications] WARNING: Do not exceed recommended intake. Consult your physicial prior to using this product if you are pregnant, nursing, taking medication, have high blood pressure, heart problems or have a medical condition. Discontinue use two weeks prior to surgery.
[Storage] Store in a cool, dry place.
[General Statements: All Other Content] Supports Memory Retention*
[Suggested/Recommended/Usage/Directions] Directions: As a dietary supplement, take one to two capsules daily.


In [12]:
[l.dsld_id for l in loader.load_flattened(order="random", seed=42)][:10]

[240944, 180818, 261493, 266151, 9232, 176267, 18040, 237604, 64455, 183189]

## load_gold — the reference reading

In [13]:
gold = loader.load_gold()
len(gold)

100

In [14]:
gold[0]

GoldLabel(dsld_id=709, name='Huperzine A', purpose=[Claim(text='supports memory retention', evidence='Supports Memory Retention*', grounded=True)], risk=[Claim(text='do not exceed recommended intake', evidence='WARNING: Do not exceed recommended intake. Consult your physicial prior to using this product if you are pregnant, nursing, taking medication, have high blood pressure, heart problems or have a medical condition. Discontinue use two weeks prior to surgery.', grounded=True), Claim(text='consult a physician before use if pregnant, nursing, taking medication, or with high blood pressure, heart problems or a medical condition', evidence='WARNING: Do not exceed recommended intake. Consult your physicial prior to using this product if you are pregnant, nursing, taking medication, have high blood pressure, heart problems or have a medical condition. Discontinue use two weeks prior to surgery.', grounded=True), Claim(text='discontinue use two weeks prior to surgery', evidence='WARNING: 

In [15]:
loader.gold_by_id()[709]

GoldLabel(dsld_id=709, name='Huperzine A', purpose=[Claim(text='supports memory retention', evidence='Supports Memory Retention*', grounded=True)], risk=[Claim(text='do not exceed recommended intake', evidence='WARNING: Do not exceed recommended intake. Consult your physicial prior to using this product if you are pregnant, nursing, taking medication, have high blood pressure, heart problems or have a medical condition. Discontinue use two weeks prior to surgery.', grounded=True), Claim(text='consult a physician before use if pregnant, nursing, taking medication, or with high blood pressure, heart problems or a medical condition', evidence='WARNING: Do not exceed recommended intake. Consult your physicial prior to using this product if you are pregnant, nursing, taking medication, have high blood pressure, heart problems or have a medical condition. Discontinue use two weeks prior to surgery.', grounded=True), Claim(text='discontinue use two weeks prior to surgery', evidence='WARNING: 

## load — what a run processes

In [16]:
loader.load().summary()

'1000 to do of 1000 available'

In [17]:
loader.load(limit=3).ids

[709, 885, 889]

In [18]:
loader.load(ids=[889, 709]).ids

[709, 889]

In [19]:
loader.load(ids=[889, 709, 99999999]).missing_ids

[99999999]

In [20]:
loader.load(limit=5, order="random", seed=42).ids

[240944, 180818, 261493, 266151, 9232]

## dump_ids / dump_path

In [21]:
ids = loader.dump_ids()
len(ids), ids[:5]

(214780, [542, 543, 544, 545, 546])

In [22]:
loader.dump_path(709)

PosixPath('/var/tmp/ge74wan/repos/supplement-kg/datasets/DSLD-full-database-JSON/000000-000999/709.json')

## errors

In [23]:
loader.load_flattened(order="random")   # no seed

ValueError: a random draw needs an explicit seed, or it cannot be redrawn: pass seed=... (any int). An unseeded draw would make every rerun score a different subset while looking identical.

In [24]:
loader.load_raw(99999999)   # not in the dump

DSLDNotFound: 99999999 is not in the dump at /var/tmp/ge74wan/repos/supplement-kg/datasets/DSLD-full-database-JSON

In [25]:
loader.load_raw([709, 99999999], skip_missing=True)   # reads the rest

[{'src': '01-raw/2023-02-23/data/label_160.json',
  'id': 709,
  'nhanesId': '',
  'bundleName': '',
  'fullName': 'Huperzine A',
  'brandName': 'GNC Herbal Plus Standardized',
  'brandIpSymbol': '®',
  'upcSku': '0 48107 06464 8',
  'productVersionCode': 'CODE 196311 HLG',
  'pdf': '',
  'thumbnail': '',
  'servingsPerContainer': None,
  'hasOuterCarton': False,
  'percentDvFootnote': 'Not Present',
  'labelRelationships': [{'type': 'Image difference, same product',
    'labelId': 63846},
   {'type': 'Image difference, same product', 'labelId': 74667},
   {'type': 'Image difference, same product', 'labelId': 243149}],
  'contacts': [{'contactId': 1193,
    'text': 'For more information',
    'types': ['Other'],
    'contactDetails': {'src': '01-raw/2025-09-25/data/contact_1.json',
     'id': 1193,
     'name': '',
     'streetAddress': '',
     'city': '',
     'state': '',
     'country': '',
     'zipCode': '',
     'phoneNumber': '1-888-462-2548',
     'email': '',
     'webAddress